# Non-Linear Regression — Practice Set 2 🎯
## Mr Beast YouTube Videos Dataset

### Rules
- `alpha = 0.05`, `random_state = 617`, 80/20 split
- Use `train` unless stated otherwise
- Do NOT round; no commas; lead with 0

### About the Data
`mr_beast.csv` — YouTube statistics for Mr Beast's videos.
- `likes`: Number of likes
- `views`: Number of views  
- `time_since`: Hours since release
- `number_of_tags`, `duration_seconds`, `length_title`, `length_description`

> The relationship between `time_since` and `likes` may not be linear. We will investigate non-linear approaches.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.formula.api import ols
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

data = pd.read_csv('mr_beast.csv')
train = data.sample(frac=0.8, random_state=617)
test  = data.drop(labels=train.index)
print('Train:', train.shape, '| Test:', test.shape)

---
## ❓ Question 1
> Read `mr_beast.csv`. Split 80% train / 20% test using `data.sample()` with `random_state=617`.
>
> **What is the mean of `time_since` in the train sample?**

In [ ]:
print('Mean time_since in train:', train.time_since.mean())

---
## ❓ Question 2
> Construct a scatter plot with `time_since` on the x-axis and `likes` on the y-axis.
> **Does the relationship look linear? Describe the pattern.**

In [ ]:
sns.scatterplot(data=train, x='time_since', y='likes', s=10, alpha=0.5)
plt.title('time_since vs likes')
plt.show()

---
## ❓ Question 3
> Fit three models predicting `likes` from `time_since`:
> - `model1`: Linear (`likes ~ time_since`)
> - `model2`: Degree-2 polynomial
> - `model3`: Degree-3 polynomial
>
> **What is the R² of `model2`?**

In [ ]:
model1 = ols('likes ~ time_since', data=train).fit()
model2 = ols('likes ~ time_since + I(time_since**2)', data=train).fit()
model3 = ols('likes ~ time_since + I(time_since**2) + I(time_since**3)', data=train).fit()

print('R² model1:', model1.rsquared)
print('R² model2:', model2.rsquared)
print('R² model3:', model3.rsquared)

---
## ❓ Question 4
> Based on `model2`, **is the quadratic term (`I(time_since**2)`) statistically significant at α=0.05?**
>
> **What is its p-value?**

In [ ]:
print(model2.summary())
print('p-value of time_since²:', model2.pvalues['I(time_since ** 2)'])

---
## ❓ Question 5
> Compare train and test RMSE across `model1`, `model2`, `model3`.
>
> **Which model has the lowest test RMSE?**
> **Is there evidence of overfitting in `model3`?**

In [ ]:
def rmse(y, yhat): return np.sqrt(np.mean((y - yhat)**2))

rows = []
for name, m in [('model1',model1), ('model2',model2), ('model3',model3)]:
    rows.append({
        'model': name,
        'train_rmse': rmse(train.likes, m.predict()),
        'test_rmse' : rmse(test.likes,  m.predict(test))
    })
df_res = pd.DataFrame(rows).set_index('model')
print(df_res)
print('\nBest test RMSE:', df_res.test_rmse.idxmin())

---
## ❓ Question 6
> Create a **step function** by binning `time_since` into:
> `[0, 10000)`, `[10000, 30000)`, `[30000, 60000)`, `[60000+)`
>
> Label them `new`, `mid`, `old`, `very_old`. Fit `likes ~ time_bin`. Call this `model_step`.
>
> **What is the R² of `model_step`?**

In [ ]:
train = train.copy(); test = test.copy()
bins = [0, 10000, 30000, 60000, float('inf')]
labels = ['new', 'mid', 'old', 'very_old']

train['time_bin'] = pd.cut(train.time_since, bins=bins, labels=labels)
test['time_bin']  = pd.cut(test.time_since,  bins=bins, labels=labels)

model_step = ols('likes ~ time_bin', data=train).fit()
print('Step R²:', model_step.rsquared)
print(model_step.params)

---
## ❓ Question 7
> Based on `model_step`, on average how many fewer likes do `very_old` videos get compared to `new` videos?
>
> (Note: `new` is the reference since it is listed first.)

In [ ]:
model_step.params['time_bin[T.very_old]']
# Negative = very_old gets fewer likes than new

---
## ❓ Question 8
> Use sklearn's `PolynomialFeatures(degree=2)` on `time_since` only.
> Fit a `LinearRegression`. Call it `sk_poly2`.
>
> **What is the predicted number of likes for a video that has been available for `20000` hours?**

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_tr = poly.fit_transform(train[['time_since']])
X_te = poly.transform(test[['time_since']])

sk_poly2 = LinearRegression().fit(X_tr, train.likes)

# Predict for time_since = 20000
new_obs = poly.transform([[20000]])
print('Predicted likes for time_since=20000:', sk_poly2.predict(new_obs)[0])

---
## ❓ Question 9
> Now use `PolynomialFeatures(degree=2)` on BOTH `time_since` AND `number_of_tags`.
> This will generate polynomial terms AND interaction terms for both variables.
>
> Fit a `LinearRegression`. Call it `sk_poly2_multi`.
> **How many features are in the expanded feature matrix?** (i.e., how many columns does it have?)
>
> **What is the train RMSE?**

In [ ]:
poly2 = PolynomialFeatures(degree=2, include_bias=False)
X_tr2 = poly2.fit_transform(train[['time_since', 'number_of_tags']])
X_te2 = poly2.transform(test[['time_since', 'number_of_tags']])

print('Feature names:', poly2.get_feature_names_out())
print('Number of features:', X_tr2.shape[1])

sk_poly2_multi = LinearRegression().fit(X_tr2, train.likes)
print('Train RMSE:', root_mean_squared_error(train.likes, sk_poly2_multi.predict(X_tr2)))
print('Test RMSE: ', root_mean_squared_error(test.likes,  sk_poly2_multi.predict(X_te2)))

---
## ❓ Question 10
> Compare the following models by test RMSE:
> - `model1`: Linear (`time_since` only)
> - `model2`: Poly-2 (`time_since` only, statsmodels)
> - `model_step`: Step function
> - `sk_poly2`: Poly-2 sklearn (time_since only)
> - `sk_poly2_multi`: Poly-2 sklearn (time_since + number_of_tags)
>
> **Which model is best? Which is worst?**

In [ ]:
comparison = pd.DataFrame({
    'model':['linear','poly2_sm','step','poly2_sk','poly2_sk_multi'],
    'test_rmse':[
        rmse(test.likes, model1.predict(test)),
        rmse(test.likes, model2.predict(test)),
        rmse(test.likes, model_step.predict(test)),
        root_mean_squared_error(test.likes, sk_poly2.predict(X_te)),
        root_mean_squared_error(test.likes, sk_poly2_multi.predict(X_te2)),
    ]
}).set_index('model').sort_values('test_rmse')
print(comparison)
print('\nBest model:', comparison.index[0])
print('Worst model:', comparison.index[-1])

---
# ✅ Answer Key & Concepts

| Q | What to Check |
|---|---|
| 1 | `train.time_since.mean()` |
| 2 | Negative slope, possibly curved |
| 3 | `model2.rsquared` |
| 4 | `model2.pvalues['I(time_since ** 2)']` |
| 5 | Compare test RMSEs; overfitting if poly3 train << test |
| 6 | `model_step.rsquared` |
| 7 | `model_step.params['time_bin[T.very_old]']` |
| 8 | `sk_poly2.predict(poly.transform([[20000]]))[0]` |
| 9 | 5 features: x1, x2, x1², x1·x2, x2² |
| 10 | Rank by test RMSE |

## 🔑 Exam Traps to Remember
- `PolynomialFeatures` on 2 variables with degree=2 creates 5 features (x1, x2, x1², x1·x2, x2²)
- `pd.cut()` with `float('inf')` as upper bound catches all higher values
- Step functions are just categorical variables — interpret coefficients vs reference group
- Adding more polynomial terms always improves train RMSE but may hurt test RMSE